# ShortPulse: training the bot filter and the URL checker

This notebook trains the two scikit-learn models the Flask app loads with joblib:

| Model | Algorithm | Used where | Decision |
|---|---|---|---|
| Bot click filter | Random Forest | click-log worker thread | is this visit automated? |
| Malicious-URL checker | Logistic Regression | link wizard, CSV import, edits | warn / require confirmation |

It calls the same functions as `ml/train_models.py` (in `ml/training.py`), and features come from `app/features.py`, which the app also uses at serving time, so training and serving can't drift apart.

**Read this before quoting any number.** By default both models train on *synthetic* data from generators in `training.py`. The metrics below show the pipeline works; they are **not** evidence of real-world accuracy, because the model is largely learning the generator's assumptions. Set `BOT_CSV` / `URL_CSV` below to train on real labelled data.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "ml")]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import training as T
from app import features as F

BOT_CSV = None   # e.g. "data/labelled_clicks.csv" with BOT_FEATURES columns + label
URL_CSV = None   # e.g. "data/urls.csv" with url,label (or url,type)
pd.set_option("display.width", 140)

## 1. Bot click data

Self-declared crawlers and link-preview fetchers (WhatsApp, Slackbot, Googlebot…) are caught by a deterministic rule in the redirect path, not by this model. The model's job is the harder case: automated traffic that looks like a browser or a script without announcing itself. That's why the features are mostly **behavioural**: clicks from the same visitor in the last 60 seconds, how many different links, and the gap since the previous click.

In [ ]:
bot_df = T.load_bot_csv(BOT_CSV) if BOT_CSV else T.synthetic_clicks()
print(len(bot_df), "rows")
print(bot_df["label"].value_counts(normalize=True).rename({0: "human", 1: "bot"}).round(3))
bot_df.head()

In [ ]:
# What separates the classes? Medians per class.
bot_df.groupby("label")[F.BOT_FEATURES].median().T.rename(columns={0: "human", 1: "bot"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label, name in ((0, "human"), (1, "bot")):
    sub = bot_df[bot_df.label == label]
    axes[0].hist(sub.clicks_last_60s.clip(upper=60), bins=30, alpha=.6, label=name)
    axes[1].hist(np.log10(sub.secs_since_prev_click + 0.1), bins=30, alpha=.6, label=name)
axes[0].set_title("clicks in last 60 s (clipped at 60)")
axes[1].set_title("log10 seconds since previous click")
axes[0].legend(); plt.tight_layout(); plt.show()

### Leakage check

* The split is stratified and random. **With real logs, split by time (train on earlier weeks, test on later) and by visitor**, otherwise the same visitor's clicks land in both train and test and the score is inflated.
* The decision threshold is picked on the validation split only (precision ≥ 0.95, because a false positive silently removes a real person's click from analytics). Test numbers are reported once, at that threshold.
* A hand-written rule (the app's fallback when no model is loaded) is the baseline. If the model can't beat it on your data, ship the rule.

In [ ]:
bot = T.train_bot_model(bot_df, trained_on="csv" if BOT_CSV else "synthetic")
pd.DataFrame({"random_forest": bot["metrics"]["model"], "rule_baseline": bot["metrics"]["baseline_rule"]}).drop(index="confusion")

In [ ]:
pd.DataFrame({"random_forest": bot["metrics"]["model"]["confusion"],
              "rule_baseline": bot["metrics"]["baseline_rule"]["confusion"]})

In [ ]:
imp = pd.Series(bot["metrics"]["feature_importance"]).sort_values()
imp.plot.barh(figsize=(7, 4), title="Random Forest feature importance"); plt.tight_layout(); plt.show()

### Failure analysis: which bots get through?

Look at the bots the model misses on the test split. In the synthetic data these are mostly the deliberately hard "low-and-slow" bots (browser user agent, a few clicks, minutes apart), which per-click features can't separate from people. Catching them would need longer-horizon signals (per-visitor history across days, IP reputation), which is a roadmap item, not a tuning problem.

In [ ]:
X_tr, y_tr, X_val, y_val, X_te, y_te = T.split(bot_df)
test = pd.DataFrame(X_te, columns=F.BOT_FEATURES).assign(label=y_te, proba=bot["model"].predict_proba(X_te)[:, 1])
missed = test[(test.label == 1) & (test.proba < bot["threshold"])]
false_alarms = test[(test.label == 0) & (test.proba >= bot["threshold"])]
print(f"missed bots: {len(missed)}   false alarms: {len(false_alarms)}")
missed[["ua_has_browser_token", "has_accept_language", "clicks_last_60s", "distinct_links_last_60s", "secs_since_prev_click", "proba"]].describe().round(2)

## 2. Malicious-URL data

Lexical features only (length, digits, IP host, `@`, suspicious words and TLDs, punycode, entropy…). No network calls: scoring is instant and can't be abused to make the server fetch arbitrary URLs. The trade-off is that a phishing page on a clean-looking domain is invisible to this model; the admin blocklist and takedown cover that case.

For real data, combine a phishing/malware feed with a list of popular benign domains, deduplicate, and check each source's licence. Keep URLs from the same domain on the same side of the split, or the model memorises domains.

In [ ]:
raw_urls = T.load_url_csv(URL_CSV) if URL_CSV else T.synthetic_urls()
print(len(raw_urls), "URLs;", raw_urls.label.mean().round(3), "malicious share")
raw_urls.groupby("label").head(3)

In [ ]:
url_df = T.url_matrix(raw_urls)
url = T.train_url_model(url_df, trained_on="csv" if URL_CSV else "synthetic")
pd.DataFrame({"logistic_regression": url["metrics"]["model"], "rule_baseline": url["metrics"]["baseline_rule"]}).drop(index="confusion")

In [ ]:
coef = pd.Series(url["metrics"]["coefficients"]).sort_values()
coef.plot.barh(figsize=(7, 6), title="Logistic Regression coefficients (standardised features)"); plt.tight_layout(); plt.show()

Coefficients on correlated features (host length, dots, hyphens, subdomains) can take surprising signs. That's collinearity, not a law of phishing; regularisation (`C=0.1`) keeps it in check. The app explains each warning with the features that pushed *this* URL's score up (coefficient × standardised value), not with raw coefficients.

In [ ]:
pipe = url["model"]
for u in ["https://docs.python.org/3/tutorial/",
          "https://www.amazon.in/dp/B0CX23V2ZK?th=1",
          "http://paytm-secure-update.xyz/login/verify",
          "http://203.0.113.9/secure/account/index.php",
          "https://campus-events.in/register"]:
    p = pipe.predict_proba(np.array([F.url_features(u)]))[0, 1]
    print(f"{p:6.3f}  {'WARN' if p >= url['threshold'] else 'ok  '}  {u}")

## 3. Save for the app

`app/ml.py` checks the saved feature list against `app/features.py` at startup and falls back to the rule baseline if they differ. Pin the scikit-learn version in `requirements.txt` to the one used here; joblib files are not portable across major versions.

In [ ]:
print(T.save(bot, "bot_model.joblib"))
print(T.save(url, "url_model.joblib"))